# *Many-Time* Pad

## Aim of this demo

This notebook demonstrates **One-Time Pad (OTP) encryption** and why reusing the same key for two or more messages is dangerous.

You will:
1. Convert text into hexadecimal (hex) representation.
2. Encrypt and decrypt a message using XOR.
3. Observe what happens when the same key encrypts two different messages.
4. Try a simple known-word guess to recover parts of the original messages.

**Important:** A true One-Time Pad is secure only when the key is random, as long as the message, kept secret, and used exactly once. Reusing the key breaks this important condition.

## 1. Text and hexadecimal representation

Computers represent characters using numbers. This notebook uses each character's ASCII code and writes that number in hexadecimal.

For example, the character `o` has decimal ASCII value 111, which is hexadecimal `6f`. Hex is used here so that XOR results can be displayed as readable pairs of digits.

In [1]:
from functools import reduce

# Converts a string to a hexadecimal string
def toHex(s):
    lst = []
    for ch in s:
        hv = hex(ord(ch)).replace('0x', '')
        if len(hv) == 1:
            hv = '0' + hv
        lst.append(hv)
    return reduce(lambda x, y: x + y, lst, '')

# Converts a hexadecimal string back to text
def toStr(s):
    if len(s) % 2 != 0:
        raise ValueError("Hex string must contain an even number of digits")
    return ''.join(chr(int(s[i:i+2], 16)) for i in range(0, len(s), 2))

print('toHex("Hello World") =', toHex("Hello World"))
print('toStr("736f6d65206d657373616765") =', toStr("736f6d65206d657373616765"))

toHex("Hello World") = 48656c6c6f20576f726c64
toStr("736f6d65206d657373616765") = some message


## 2. XOR operation

XOR (exclusive OR) compares two bits:
- Same bits produce `0` (0 XOR 0 = 0, 1 XOR 1 = 0).
- Different bits produce `1` (0 XOR 1 = 1, 1 XOR 0 = 1).

A useful property is:

\[
P \oplus K = C, \qquad C \oplus K = P
\]

So the same key can encrypt and decrypt. The two hex strings must have the same length.

In [2]:
# Computes XOR of two hexadecimal strings.
# Both strings must have the same length.
def Xor(s1, s2):
    if len(s1) != len(s2):
        raise ValueError("Both inputs must have the same length")
    res = ""
    for i in range(len(s1)):
        res += format(int(s1[i], 16) ^ int(s2[i], 16), '01x')
    return res

# Small XOR check
print("5 XOR 3 =", format(0x5 ^ 0x3, 'x'))

5 XOR 3 = 6


## 3. Encrypting and decrypting one message

Encryption: convert the message and key to hex, then XOR them.

Decryption: XOR the ciphertext with the same key again, then convert the result back to text.

In [3]:
message = "secret message"
key_single = "my secret keys"

if len(message) != len(key_single):
    raise ValueError("For this demo, message and key must have equal lengths")

print("hex(message) =", toHex(message))
print("hex(key)     =", toHex(key_single))

ciphertext = Xor(toHex(message), toHex(key_single))
print("ciphertext   =", ciphertext)

recovered_message = toStr(Xor(ciphertext, toHex(key_single)))
print("recovered message =", recovered_message)

assert recovered_message == message
print("Check passed: decrypted message matches the original.")

hex(message) = 736563726574206d657373616765
hex(key)     = 6d7920736563726574206b657973
ciphertext   = 1e1c430100175208115318041e16
recovered message = secret message
Check passed: decrypted message matches the original.


## 4. What happens when the same key is reused?

Now encrypt two different messages with the **same key**. The messages are chosen to have equal lengths so that XOR can be applied position by position.

The key identity is:

\[
C_1 = P_1 \oplus K, \qquad C_2 = P_2 \oplus K
\]

XOR the ciphertexts:

\[
C_1 \oplus C_2
= (P_1 \oplus K) \oplus (P_2 \oplus K)
= P_1 \oplus P_2
\]

The key cancels out because \(K \oplus K = 0\). Therefore, an attacker who intercepts both ciphertexts can obtain the XOR of the two plaintexts without knowing the key.

In [4]:
message1 = "steal the secret"
message2 = "the cat the bird"
key = "supersecretverys"

assert len(message1) == len(message2) == len(key)

ciphertext1 = Xor(toHex(message1), toHex(key))
ciphertext2 = Xor(toHex(message2), toHex(key))

xor_ciphertexts = Xor(ciphertext1, ciphertext2)
xor_messages = Xor(toHex(message1), toHex(message2))

print("Ciphertext 1:", ciphertext1)
print("Ciphertext 2:", ciphertext2)
print("XOR of ciphertexts:", xor_ciphertexts)
print("XOR of messages:   ", xor_messages)

assert xor_ciphertexts == xor_messages
print("\nXOR of the ciphertexts is the same as XOR of the messages.")

Ciphertext 1: 000115041e53110b1745071306001c07
Ciphertext 2: 071d154511121143060d1156071b0b17
XOR of ciphertexts: 071c00410f41004811481645011b1710
XOR of messages:    071c00410f41004811481645011b1710

XOR of the ciphertexts is the same as XOR of the messages.


## 5. Trying to guess part of a message

The attacker does not immediately see the two original messages. However, the XOR of the messages can leak information.

English text often contains common words such as ` the `. Suppose we guess that one plaintext contains ` the ` at a particular position. XORing that guessed text with the corresponding part of \(P_1 \oplus P_2\) gives a candidate part of the other plaintext.

If the result looks like ordinary English letters and spaces, the guess may be correct. This is only a simple demonstration; real cryptanalysis requires testing many possibilities and using context.

In [5]:
def TryGuessingSubstring(substring, message_length, xor_messages):
    good_guesses = []
    for pos in range(message_length - len(substring) + 1):
        guess = toHex(
            chr(0) * pos
            + substring
            + chr(0) * (message_length - len(substring) - pos)
        )
        other_message_part = toStr(Xor(guess, xor_messages))[pos:pos + len(substring)]
        good_guess = True
        for ch in other_message_part:
            if not ch.isalpha() and not ch.isspace():
                good_guess = False
                break
        if good_guess:
            good_guesses.append((pos, other_message_part))

    print("\nPossible guesses:")
    for pos, other_part in good_guesses:
        print(
            'position:', pos,
            '| guessed part:', repr(substring),
            '| possible other-message part:', repr(other_part)
        )
    return good_guesses

guesses_the = TryGuessingSubstring(" the ", len(message1), xor_messages)


Possible guesses:
position: 5 | guessed part: ' the ' | possible other-message part: 'at th'
position: 7 | guessed part: ' the ' | possible other-message part: 'he se'


### Try a more specific guess

The original workshop example next tries the phrase `at the `. Compare the possible positions and use the surrounding English context to decide whether a candidate is plausible.

In [6]:
guesses_at_the = TryGuessingSubstring("at the ", len(message1), xor_messages)


Possible guesses:
position: 5 | guessed part: 'at the ' | possible other-message part: ' the se'


## 6. What did we learn?

- XOR with the same key encrypts and decrypts a message.
- If a key is reused, XORing two ciphertexts removes the key and reveals the XOR of the two plaintexts.
- Common words and language patterns can help an attacker guess parts of the plaintexts.
- Reusing a One-Time Pad key is unsafe. A true OTP key must be random, secret, message-length, and used only once.

**Conclusion:** The security of the One-Time Pad depends on never reusing its key. This is why the notebook is called *Many-Time Pad*: it demonstrates the weakness caused by using one pad many times.

In [7]:
# Final verification of the central result
assert Xor(ciphertext1, ciphertext2) == Xor(toHex(message1), toHex(message2))
assert toStr(Xor(ciphertext, toHex(key_single))) == message
print("All final verification checks passed.")

All final verification checks passed.
